In [1]:
import sys
print(sys.executable)

/home/midori/Desktop/Spam_Mail_Detection/.myenv/bin/python


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [3]:
raw_mail_data = pd.read_csv('../Data/mail_data.csv')
raw_mail_data.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
raw_mail_data.shape

(5572, 2)

#### Data preprocessing

In [5]:
# All NaN values → replaced with empty string ''
# mail_data = raw_mail_data.where(pd.notnull(raw_mail_data),'') 
# the above might effect the model negatively, TF-IDF will treat it as zero vector, 
# if found NaN just drop the row

mail_data = raw_mail_data.dropna()

In [6]:
mail_data.shape

(5572, 2)

Label encoading

In [7]:
# spam mail -> 0
# ham mail -> 1

mail_data['Category'] = mail_data['Category'].map({'spam': 0, 'ham': 1})
# any value that does not match 'spam' or 'ham' is automatically converted to NaN. its a good practice
# to allways run isnull().sum() after it

In [8]:
mail_data['Category'].isnull().sum()

np.int64(0)

sepearating X and Y

In [9]:
X = mail_data.iloc[:, 1]
X

0       Go until jurong point, crazy.. Available only ...
1                           Ok lar... Joking wif u oni...
2       Free entry in 2 a wkly comp to win FA Cup fina...
3       U dun say so early hor... U c already then say...
4       Nah I don't think he goes to usf, he lives aro...
                              ...                        
5567    This is the 2nd time we have tried 2 contact u...
5568                 Will ü b going to esplanade fr home?
5569    Pity, * was in mood for that. So...any other s...
5570    The guy did some bitching but I acted like i'd...
5571                           Rofl. Its true to its name
Name: Message, Length: 5572, dtype: str

In [10]:
# X = mail_data['Message']
# y = mail_data['Category']
# X = mail_data[['Message', 'Subject', 'Sender']]

Y = mail_data.iloc[:, 0]
Y

0       1
1       1
2       0
3       1
4       1
       ..
5567    0
5568    1
5569    1
5570    1
5571    1
Name: Category, Length: 5572, dtype: int64

Train-Test split

In [11]:
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=3)

"""
random_state is used to make experiments reproducible.

Many operations like train_test_split, shuffling, or model initialization involve randomness.
If random_state is not set, the split or results will change every time you run the code.

By setting random_state=3 (or any fixed number), you ensure:
- The same data split every run
- The same shuffling order
- Comparable and consistent results

It does NOT improve model performance.
It only controls randomness for reproducibility and debugging.
"""

'\nrandom_state is used to make experiments reproducible.\n\nMany operations like train_test_split, shuffling, or model initialization involve randomness.\nIf random_state is not set, the split or results will change every time you run the code.\n\nBy setting random_state=3 (or any fixed number), you ensure:\n- The same data split every run\n- The same shuffling order\n- Comparable and consistent results\n\nIt does NOT improve model performance.\nIt only controls randomness for reproducibility and debugging.\n'

In [12]:
print(X.shape)
print(X_train.shape)
print(X_test.shape)

(5572,)
(4457,)
(1115,)


#### Feature Extraction (TF-IDF_VECTORIZER)

In [13]:
feature_extraction = TfidfVectorizer(min_df=1,
                                     stop_words='english',
                                     lowercase=True)
X_train_features = feature_extraction.fit_transform(X_train)
X_test_features = feature_extraction.transform(X_test)

# we fit the data with the training sample once and then transform for both training and testing datas
# df = document frequency
# If you have 100 emails:
#  “free” appears in 40 emails → df = 40
#  “qwertyx” appears in 1 email → df = 1

# min_df removes words that appear in too few documents.
# max_df removes words that appear in too many documents.
# either we can assign integer or we can give number between 0-1

In [14]:
Y_train = Y_train.astype('int')
Y_test = Y_test.astype('int')

In [15]:
print(X_train_features)

# Rows = 4457 → number of training emails
# Columns = 7431 → number of unique words in vocabulary

# Each row = one email
# Each column = one word feature

# Only 34,775 are non-zero.
# Everything else is zero.


# example
# (0, 2329)  0.38783870336935383
# Row 0 (first email)
# Column 2329 (word index 2329 in vocabulary)
# TF-IDF value = 0.3878

# In email 0, word #2329 has TF-IDF value 0.3878.

# Email 0 → [0, 0, 0.38, 0, 0.41, 0, ..., 0.61, 0, 0]
# Email 1 → [0, 0.17, 0, 0.25, 0.33, ..., 0]


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 34775 stored elements and shape (4457, 7431)>
  Coords	Values
  (0, 2329)	0.38783870336935383
  (0, 3811)	0.34780165336891333
  (0, 2224)	0.413103377943378
  (0, 4456)	0.4168658090846482
  (0, 5413)	0.6198254967574347
  (1, 3811)	0.17419952275504033
  (1, 3046)	0.2503712792613518
  (1, 1991)	0.33036995955537024
  (1, 2956)	0.33036995955537024
  (1, 2758)	0.3226407885943799
  (1, 1839)	0.2784903590561455
  (1, 918)	0.22871581159877646
  (1, 2746)	0.3398297002864083
  (1, 2957)	0.3398297002864083
  (1, 3325)	0.31610586766078863
  (1, 3185)	0.29694482957694585
  (1, 4080)	0.18880584110891163
  (2, 6601)	0.6056811524587518
  (2, 2404)	0.45287711070606745
  (2, 3156)	0.4107239318312698
  (2, 407)	0.509272536051008
  (3, 7414)	0.8100020912469564
  (3, 2870)	0.5864269879324768
  (4, 2870)	0.41872147309323743
  (4, 487)	0.2899118421746198
  :	:
  (4454, 2855)	0.47210665083641806
  (4454, 2246)	0.47210665083641806
  (4455, 4456)	0.24

In [16]:
feature_names = feature_extraction.get_feature_names_out()
print(feature_names[2329])
print(feature_names[3811])
print(feature_names[2224])
print(feature_names[4456])
print(feature_names[5413])

don
know
did
msg
recently


In [17]:
print(X_train.iloc[0])

Don know. I did't msg him recently.


In [18]:
print(X_train[0])

Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...


#### Training Logistic regression

In [19]:
model = LogisticRegression()

In [20]:
model.fit(X_train_features,Y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

#### Evaluation

In [28]:
# predictions on training data
X_train_prediction = model.predict(X_train_features)

In [29]:
print(f"accuracy on training data: {accuracy_score(X_train_prediction,Y_train)*100}%")

accuracy on training data: 96.76912721561588%


In [30]:
# predictions on testing data
X_test_prediction = model.predict(X_test_features)

In [32]:
print(f"accuracy on testing data: {accuracy_score(X_test_prediction,Y_test)*100}%")

# need to check for both training and testing data accuracy to find overfitting and underfitting

accuracy on testing data: 96.68161434977578%


## Building a predictive system

In [40]:
input_mail = ["WINNER!! As a valued network customer you have been selected to receivea £900 prize reward! "]

# text -> feature vectors
input_mail_features = feature_extraction.transform(input_mail)

# making prediction
prediction = model.predict(input_mail_features)

if prediction[0]==1:
    print('Ham mail')
else:
    print('SPAM mail !!!')

# assumpting spam==0 and ham==1

SPAM mail !!!
